<a href="https://colab.research.google.com/github/rmadatt/AI-Newton-StudyGroupAfrica/blob/main/AI_Newton_Elastic_Collision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gplearn

In [ ]:
!pip install gplearn numpy pandas scikit-learn

In [5]:
import numpy as np

# Re-create the configuration and simulation to define the 'dataset'
elastic_collision_config = {
    "experiment_id": "elastic_collision_1d_001",
    "system_type": "two_body_collision",
    "parameters": {
        "time_step": 0.01,
        "total_duration": 3.0
    },
    "objects": [
        {
            "name": "Body_A",
            "mass": 2.0,
            "initial_position": 0.0,
            "initial_velocity": 4.0
        },
        {
            "name": "Body_B",
            "mass": 3.0,
            "initial_position": 8.0,
            "initial_velocity": -1.0
        }
    ]
}

def run_elastic_collision_experiment(config):
    obj_a, obj_b = config["objects"]
    m1, x1, v1 = obj_a["mass"], obj_a["initial_position"], obj_a["initial_velocity"]
    m2, x2, v2 = obj_b["mass"], obj_b["initial_position"], obj_b["initial_velocity"]
    dt = config["parameters"]["time_step"]
    duration = config["parameters"]["total_duration"]
    steps = int(duration / dt)
    observation_logs = []
    collision_occurred = False
    for step in range(steps):
        t = step * dt
        if not collision_occurred and x1 >= x2:
            v1_final = ((m1 - m2) * v1 + 2 * m2 * v2) / (m1 + m2)
            v2_final = ((m2 - m1) * v2 + 2 * m1 * v1) / (m1 + m2)
            v1, v2 = v1_final, v2_final
            collision_occurred = True
        x1 += v1 * dt
        x2 += v2 * dt
        observation_logs.append({
            "time": round(t, 3),
            "body_a_position": round(x1, 4),
            "body_a_velocity": round(v1, 4),
            "body_b_position": round(x2, 4),
            "body_b_velocity": round(v2, 4)
        })
    return observation_logs

dataset = run_elastic_collision_experiment(elastic_collision_config)

def discover_conservation_law(dataset):
    """
    Scans candidate mathematical expressions of state variables
    to find a conserved quantity (zero or near-zero variance over time).
    """
    t = np.array([row["time"] for row in dataset])
    v1 = np.array([row["body_a_velocity"] for row in dataset])
    v2 = np.array([row["body_b_velocity"] for row in dataset])
    m1, m2 = 2.0, 3.0
    candidates = {
        "Total Velocity (v1 + v2)": v1 + v2,
        "Relative Velocity (v1 - v2)": v1 - v2,
        "Sum of Velocities Squared (v1^2 + v2^2)": (v1**2) + (v2**2),
        "Linear Momentum (m1*v1 + m2*v2)": (m1 * v1) + (m2 * v2),
        "Kinetic Energy (0.5*m1*v1^2 + 0.5*m2*v2^2)": (0.5 * m1 * (v1**2)) + (0.5 * m2 * (v2**2))
    }
    results = []
    print("Evaluating candidate expressions for conservation laws...\n")
    for name, values in candidates.items():
        variance = np.var(values)
        initial_val = values[0]
        final_val = values[-1]
        is_conserved = np.isclose(initial_val, final_val, atol=1e-3) and variance < 1e-2
        results.append({
            "expression": name,
            "variance": variance,
            "conserved": is_conserved
        })
        status = "CONSERVED LAW DISCOVERED!" if is_conserved else "Rejected (Changes over time)"
        print(f"[{status}] {name} -> Variance: {variance:.5f}")
    return results

discovery_results = discover_conservation_law(dataset)

Evaluating candidate expressions for conservation laws...

[Rejected (Changes over time)] Total Velocity (v1 + v2) -> Variance: 0.99462
[Rejected (Changes over time)] Relative Velocity (v1 - v2) -> Variance: 24.86556
[Rejected (Changes over time)] Sum of Velocities Squared (v1^2 + v2^2) -> Variance: 3.97849
[CONSERVED LAW DISCOVERED!] Linear Momentum (m1*v1 + m2*v2) -> Variance: 0.00000
[CONSERVED LAW DISCOVERED!] Kinetic Energy (0.5*m1*v1^2 + 0.5*m2*v2^2) -> Variance: 0.00000
